# 4-Class Multimodal Deepfake Detection (Journal-Ready, Kaggle T4)
This notebook builds a production-grade, end-to-end multimodal system that fuses video and audio cues to detect four classes of deepfakes:
- **FF**: Fake Video + Fake Audio
- **FR**: Fake Video + Real Audio
- **RF**: Real Video + Fake Audio
- **RR**: Real Video + Real Audio

## SECTION 1: GLOBAL CONFIGURATION & DEPENDENCIES

In [ ]:
!pip -q install timm transformers librosa soundfile opencv-python-headless

In [ ]:
import os
import random
import json
import math
import glob
import shutil
import zipfile
import subprocess
import uuid
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn.utils.rnn import pad_sequence
from tqdm.auto import tqdm

import cv2
import librosa
import soundfile as sf
import timm
from transformers import Wav2Vec2Processor, Wav2Vec2Model

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupKFold
from sklearn.metrics import confusion_matrix, roc_curve, auc

In [ ]:
def set_global_seed(seed: int = 42) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

@dataclass
class Config:
    VISUAL_FPS: int = 12
    FACE_SIZE: int = 224
    AUDIO_SR: int = 16000
    VISUAL_BACKBONE: str = "efficientnetv2_rw_m"
    AUDIO_BACKBONE: str = "facebook/wav2vec2-xls-r-300m"
    PROJECTION_DIM: int = 256
    NUM_HEADS: int = 4
    BATCH_SIZE: int = 16
    EPOCHS: int = 15
    LEARNING_RATE: float = 1e-4
    WEIGHT_DECAY: float = 1e-2
    NUM_CLASSES: int = 4
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    FEATURES_DIR: str = "./extracted_features"
    MANIFEST_PATH: str = "/kaggle/input/manifest.csv"
    WORK_DIR: str = "./working_artifacts"
    RANDOM_SEED: int = 42
    AUDIO_CHUNK_SEC: float = 2.0
    MAX_VISUAL_FRAMES: int = 120
    MAX_AUDIO_CHUNKS: int = 120

cfg = Config()
set_global_seed(cfg.RANDOM_SEED)
os.makedirs(cfg.FEATURES_DIR, exist_ok=True)
os.makedirs(cfg.WORK_DIR, exist_ok=True)
print("Device:", cfg.DEVICE)

## SECTION 2: HIGH-EFFICIENCY DECOUPLED EXTRACTION PIPELINE

In [ ]:
FACE_CASCADE = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def center_crop(frame: np.ndarray, size: int) -> np.ndarray:
    h, w = frame.shape[:2]
    min_dim = min(h, w)
    start_x = (w - min_dim) // 2
    start_y = (h - min_dim) // 2
    crop = frame[start_y:start_y + min_dim, start_x:start_x + min_dim]
    return cv2.resize(crop, (size, size))

def extract_primary_face(frame: np.ndarray, size: int) -> np.ndarray:
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = FACE_CASCADE.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4)
    if len(faces) == 0:
        return center_crop(frame, size)
    x, y, w, h = max(faces, key=lambda b: b[2] * b[3])
    face = frame[y:y + h, x:x + w]
    return cv2.resize(face, (size, size))

def sample_video_frames(video_path: str, fps: int, size: int, max_frames: int) -> torch.Tensor:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {video_path}")
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    if not video_fps or math.isnan(video_fps):
        video_fps = fps
    frame_interval = max(int(round(video_fps / fps)), 1)
    frames = []
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % frame_interval == 0:
            face = extract_primary_face(frame, size)
            face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
            face = face.astype(np.float32) / 255.0
            frames.append(torch.from_numpy(face).permute(2, 0, 1))
            if len(frames) >= max_frames:
                break
        idx += 1
    cap.release()
    if len(frames) == 0:
        raise RuntimeError(f"No frames sampled from: {video_path}")
    return torch.stack(frames)

def extract_audio_to_wav(video_path: str, output_wav: str, sr: int) -> None:
    cmd = [
        "ffmpeg", "-y", "-i", video_path,
        "-vn", "-ac", "1", "-ar", str(sr),
        "-loglevel", "error", output_wav
    ]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if result.returncode != 0:
        raise RuntimeError(f"FFmpeg audio extraction failed for {video_path}: {result.stderr.decode('utf-8')}")

def load_audio_chunks(video_path: str, sr: int, chunk_sec: float, max_chunks: int) -> torch.Tensor:
    temp_wav = os.path.join(cfg.WORK_DIR, f"temp_audio_{uuid.uuid4().hex}.wav")
    try:
        extract_audio_to_wav(video_path, temp_wav, sr)
        audio, _ = librosa.load(temp_wav, sr=sr, mono=True)
        if len(audio) == 0:
            raise RuntimeError(f"Empty audio for: {video_path}")
        chunk_len = int(sr * chunk_sec)
        chunks = []
        for start in range(0, len(audio), chunk_len):
            segment = audio[start:start + chunk_len]
            if len(segment) < chunk_len:
                segment = np.pad(segment, (0, chunk_len - len(segment)))
            chunks.append(torch.from_numpy(segment))
            if len(chunks) >= max_chunks:
                break
        return torch.stack(chunks)
    finally:
        if os.path.exists(temp_wav):
            os.remove(temp_wav)

def build_visual_backbone(name: str) -> nn.Module:
    model = timm.create_model(name, pretrained=True, num_classes=0, global_pool="")
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    return model

def build_audio_backbone(name: str) -> Tuple[Wav2Vec2Processor, Wav2Vec2Model]:
    processor = Wav2Vec2Processor.from_pretrained(name)
    model = Wav2Vec2Model.from_pretrained(name)
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    return processor, model

@torch.no_grad()
def extract_multimodal_features(video_path: str, visual_model: nn.Module, audio_processor: Wav2Vec2Processor, audio_model: Wav2Vec2Model) -> Dict[str, torch.Tensor]:
    frames = sample_video_frames(video_path, cfg.VISUAL_FPS, cfg.FACE_SIZE, cfg.MAX_VISUAL_FRAMES)
    frames = frames.to(cfg.DEVICE)
    mean = torch.tensor([0.485, 0.456, 0.406], device=cfg.DEVICE).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=cfg.DEVICE).view(1, 3, 1, 1)
    frames = (frames - mean) / std
    visual_features = []
    batch_size = 16
    for i in range(0, frames.size(0), batch_size):
        batch = frames[i:i + batch_size]
        feats = visual_model(batch)
        if feats.ndim == 4:
            feats = feats.mean(dim=[2, 3])
        visual_features.append(feats.cpu())
    visual_feats = torch.cat(visual_features, dim=0)

    audio_chunks = load_audio_chunks(video_path, cfg.AUDIO_SR, cfg.AUDIO_CHUNK_SEC, cfg.MAX_AUDIO_CHUNKS)
    audio_features = []
    for i in range(audio_chunks.size(0)):
        inputs = audio_processor(audio_chunks[i].numpy(), sampling_rate=cfg.AUDIO_SR, return_tensors="pt", padding=True)
        inputs = {k: v.to(cfg.DEVICE) for k, v in inputs.items()}
        outputs = audio_model(**inputs)
        hidden = outputs.last_hidden_state.mean(dim=1)
        audio_features.append(hidden.cpu())
    audio_feats = torch.cat(audio_features, dim=0)

    return {"visual_feats": visual_feats, "audio_feats": audio_feats}

def extract_and_save_features(video_paths: List[str]) -> None:
    visual_model = build_visual_backbone(cfg.VISUAL_BACKBONE).to(cfg.DEVICE)
    audio_processor, audio_model = build_audio_backbone(cfg.AUDIO_BACKBONE)
    audio_model = audio_model.to(cfg.DEVICE)

    for video_path in tqdm(video_paths, desc="Extracting features"):
        file_id = os.path.splitext(os.path.basename(video_path))[0]
        save_path = os.path.join(cfg.FEATURES_DIR, f"{file_id}.pt")
        if os.path.exists(save_path):
            continue
        feats = extract_multimodal_features(video_path, visual_model, audio_processor, audio_model)
        torch.save(feats, save_path)

## SECTION 3: COMPACT DATASET & VARIABLE-LENGTH PADDING

In [ ]:
class FeatureDataset(Dataset):
    def __init__(self, manifest: List[Dict[str, Any]], features_dir: str):
        self.manifest = manifest
        self.features_dir = features_dir

    def __len__(self) -> int:
        return len(self.manifest)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        item = self.manifest[idx]
        file_id = item["file_id"]
        feature_path = os.path.join(self.features_dir, f"{file_id}.pt")
        if not os.path.exists(feature_path):
            raise FileNotFoundError(f"Missing features for {file_id} at {feature_path}")
        feats = torch.load(feature_path, map_location="cpu")
        return {
            "visual_feats": feats["visual_feats"],
            "audio_feats": feats["audio_feats"],
            "label": int(item["label"]),
            "identity": item["identity"]
        }

def pad_multimodal_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    visual = [b["visual_feats"] for b in batch]
    audio = [b["audio_feats"] for b in batch]
    labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    visual_pad = pad_sequence(visual, batch_first=True)
    audio_pad = pad_sequence(audio, batch_first=True)
    visual_len = torch.tensor([v.shape[0] for v in visual], dtype=torch.long)
    audio_len = torch.tensor([a.shape[0] for a in audio], dtype=torch.long)
    return {
        "visual": visual_pad,
        "audio": audio_pad,
        "visual_len": visual_len,
        "audio_len": audio_len,
        "labels": labels
    }

def load_manifest(csv_path: str) -> List[Dict[str, Any]]:
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Manifest not found at {csv_path}")
    import pandas as pd
    df = pd.read_csv(csv_path)
    required = {"file_id", "label", "identity"}
    if not required.issubset(df.columns):
        raise ValueError(f"Manifest must contain columns: {required}")
    return df[["file_id", "label", "identity"]].astype({"file_id": str, "label": int}).to_dict(orient="records")

## SECTION 4: CROSS-MODAL ATTENTION FUSION NETWORK (MATHEMATICAL CORE)

In [ ]:
class CrossModalFusionNet(nn.Module):
    def __init__(self, visual_dim: int, audio_dim: int, proj_dim: int, num_heads: int, num_classes: int):
        super().__init__()
        self.visual_proj = nn.Linear(visual_dim, proj_dim)
        self.audio_proj = nn.Linear(audio_dim, proj_dim)
        self.attn_vq = nn.MultiheadAttention(embed_dim=proj_dim, num_heads=num_heads, batch_first=True)
        self.attn_aq = nn.MultiheadAttention(embed_dim=proj_dim, num_heads=num_heads, batch_first=True)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim * 2, proj_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(proj_dim, num_classes)
        )

    def forward(self, visual: torch.Tensor, audio: torch.Tensor) -> torch.Tensor:
        v = self.visual_proj(visual)
        a = self.audio_proj(audio)
        v_attn, _ = self.attn_vq(query=v, key=a, value=a)
        a_attn, _ = self.attn_aq(query=a, key=v, value=v)
        v_pool = self.pool(v_attn.transpose(1, 2)).squeeze(-1)
        a_pool = self.pool(a_attn.transpose(1, 2)).squeeze(-1)
        fused = torch.cat([v_pool, a_pool], dim=-1)
        return self.classifier(fused)

def infer_feature_dims(sample_feature_path: str) -> Tuple[int, int]:
    feats = torch.load(sample_feature_path, map_location="cpu")
    visual_dim = feats["visual_feats"].shape[-1]
    audio_dim = feats["audio_feats"].shape[-1]
    return visual_dim, audio_dim

## SECTION 5: IDENTITY-AWARE CROSS-VALIDATION & TRAINING LOOP

In [ ]:
def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: AdamW, criterion: nn.Module) -> Tuple[float, float]:
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    progress = tqdm(loader, desc="Train", leave=False)
    for batch in progress:
        visual = batch["visual"].to(cfg.DEVICE)
        audio = batch["audio"].to(cfg.DEVICE)
        labels = batch["labels"].to(cfg.DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(visual, audio)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        progress.set_postfix({"loss": total_loss / total, "acc": correct / total})
    return total_loss / total, correct / total

@torch.no_grad()
def validate_one_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> Tuple[float, float, List[int], List[int], List[np.ndarray]]:
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_labels, all_preds, all_probs = [], [], []
    progress = tqdm(loader, desc="Val", leave=False)
    for batch in progress:
        visual = batch["visual"].to(cfg.DEVICE)
        audio = batch["audio"].to(cfg.DEVICE)
        labels = batch["labels"].to(cfg.DEVICE)

        logits = model(visual, audio)
        loss = criterion(logits, labels)
        probs = F.softmax(logits, dim=1).detach().cpu().numpy()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_labels.extend(labels.cpu().tolist())
        all_preds.extend(preds.cpu().tolist())
        all_probs.extend(probs)
        progress.set_postfix({"loss": total_loss / total, "acc": correct / total})
    return total_loss / total, correct / total, all_labels, all_preds, all_probs

def run_identity_aware_cv(manifest: List[Dict[str, Any]]) -> Dict[str, Any]:
    file_ids = [m["file_id"] for m in manifest]
    groups = [m["identity"] for m in manifest]
    labels = [m["label"] for m in manifest]

    gkf = GroupKFold(n_splits=5)
    metrics = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    fold_reports = []

    sample_feature = os.path.join(cfg.FEATURES_DIR, f"{file_ids[0]}.pt")
    visual_dim, audio_dim = infer_feature_dims(sample_feature)

    for fold, (train_idx, val_idx) in enumerate(gkf.split(file_ids, labels, groups), start=1):
        train_manifest = [manifest[i] for i in train_idx]
        val_manifest = [manifest[i] for i in val_idx]

        train_loader = DataLoader(FeatureDataset(train_manifest, cfg.FEATURES_DIR), batch_size=cfg.BATCH_SIZE, shuffle=True, collate_fn=pad_multimodal_collate, num_workers=2)
        val_loader = DataLoader(FeatureDataset(val_manifest, cfg.FEATURES_DIR), batch_size=cfg.BATCH_SIZE, shuffle=False, collate_fn=pad_multimodal_collate, num_workers=2)

        model = CrossModalFusionNet(visual_dim, audio_dim, cfg.PROJECTION_DIM, cfg.NUM_HEADS, cfg.NUM_CLASSES).to(cfg.DEVICE)
        optimizer = AdamW(model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss()

        for epoch in range(cfg.EPOCHS):
            train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
            val_loss, val_acc, all_labels, all_preds, all_probs = validate_one_epoch(model, val_loader, criterion)

            metrics["train_loss"].append(train_loss)
            metrics["train_acc"].append(train_acc)
            metrics["val_loss"].append(val_loss)
            metrics["val_acc"].append(val_acc)

            print(f"Fold {fold} | Epoch {epoch+1}/{cfg.EPOCHS} | Train Loss {train_loss:.4f} Acc {train_acc:.4f} | Val Loss {val_loss:.4f} Acc {val_acc:.4f}")

        fold_reports.append({
            "model_state": model.state_dict(),
            "labels": all_labels,
            "preds": all_preds,
            "probs": all_probs,
            "val_manifest": val_manifest
        })

    return {"metrics": metrics, "fold_reports": fold_reports}

## SECTION 6: RIGOROUS DATA VISUALIZATION & OUTPUT PACKAGING

In [ ]:
def plot_training_curves(metrics: Dict[str, List[float]]) -> str:
    epochs = list(range(1, len(metrics["train_loss"]) + 1))
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax1.set_title("Loss and Accuracy Convergence")
    ax1.plot(epochs, metrics["train_loss"], label="Train Loss", color="tab:blue")
    ax1.plot(epochs, metrics["val_loss"], label="Val Loss", color="tab:orange")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper left")

    ax2 = ax1.twinx()
    ax2.plot(epochs, metrics["train_acc"], label="Train Acc", color="tab:green")
    ax2.plot(epochs, metrics["val_acc"], label="Val Acc", color="tab:red")
    ax2.set_ylabel("Accuracy")
    ax2.legend(loc="upper right")
    fig.tight_layout()
    out_path = os.path.join(cfg.WORK_DIR, "training_curves.png")
    fig.savefig(out_path, dpi=300)
    plt.show()
    return out_path

def plot_confusion_matrix(labels: List[int], preds: List[int]) -> str:
    cm = confusion_matrix(labels, preds, labels=[0, 1, 2, 3])
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, xticklabels=["FF", "FR", "RF", "RR"], yticklabels=["FF", "FR", "RF", "RR"])
    ax.set_title("4-Class Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    fig.tight_layout()
    out_path = os.path.join(cfg.WORK_DIR, "confusion_matrix.png")
    fig.savefig(out_path, dpi=300)
    plt.show()
    return out_path

def plot_multiclass_roc(labels: List[int], probs: List[np.ndarray]) -> str:
    labels_np = np.array(labels)
    probs_np = np.array(probs)
    fig, ax = plt.subplots(figsize=(7, 6))
    for class_idx, class_name in enumerate(["FF", "FR", "RF", "RR"]):
        fpr, tpr, _ = roc_curve(labels_np == class_idx, probs_np[:, class_idx])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f"{class_name} (AUC={roc_auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--")
    ax.set_title("Multiclass ROC Curves")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(loc="lower right")
    fig.tight_layout()
    out_path = os.path.join(cfg.WORK_DIR, "roc_curves.png")
    fig.savefig(out_path, dpi=300)
    plt.show()
    return out_path

def package_outputs(model_path: str, artifacts: List[str]) -> str:
    zip_path = os.path.join(cfg.WORK_DIR, "deepfake_multimodal_outputs.zip")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(model_path, arcname=os.path.basename(model_path))
        for art in artifacts:
            zf.write(art, arcname=os.path.basename(art))
    return zip_path

## SECTION 7: END-TO-END EVALUATION INFERENCE PIPELINE

In [ ]:
@torch.no_grad()
def classify_testing_video(video_path: str, model_path: str) -> Dict[str, float]:
    visual_model = build_visual_backbone(cfg.VISUAL_BACKBONE).to(cfg.DEVICE)
    audio_processor, audio_model = build_audio_backbone(cfg.AUDIO_BACKBONE)
    audio_model = audio_model.to(cfg.DEVICE)

    feats = extract_multimodal_features(video_path, visual_model, audio_processor, audio_model)
    visual = feats["visual_feats"].unsqueeze(0).to(cfg.DEVICE)
    audio = feats["audio_feats"].unsqueeze(0).to(cfg.DEVICE)

    visual_dim = visual.shape[-1]
    audio_dim = audio.shape[-1]
    model = CrossModalFusionNet(visual_dim, audio_dim, cfg.PROJECTION_DIM, cfg.NUM_HEADS, cfg.NUM_CLASSES).to(cfg.DEVICE)
    model.load_state_dict(torch.load(model_path, map_location=cfg.DEVICE))
    model.eval()

    logits = model(visual, audio)
    probs = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    classes = ["FF", "FR", "RF", "RR"]
    results = {cls: float(prob * 100) for cls, prob in zip(classes, probs)}

    pred = classes[int(np.argmax(probs))]
    verdict = "Authentic (RR)" if pred == "RR" else f"Manipulated ({pred})"
    print("==== Deepfake Classification Summary ====")
    for cls in classes:
        print(f"{cls}: {results[cls]:.2f}%")
    print(f"Verdict: {verdict}")
    return results

---
## End-to-End Execution Example
Update dataset paths to match your Kaggle inputs, then run cells in order.

In [ ]:
# 1) Load manifest
manifest = load_manifest(cfg.MANIFEST_PATH)

# 2) Extract features (supply your MP4 paths)
video_paths = [os.path.join("/kaggle/input/videos", f"{m['file_id']}.mp4") for m in manifest]
extract_and_save_features(video_paths)

# 3) Train with identity-aware cross-validation
results = run_identity_aware_cv(manifest)
metrics = results["metrics"]
fold_reports = results["fold_reports"]

# 4) Persist the last fold (replace with your selection strategy)
best_model_path = os.path.join(cfg.WORK_DIR, "fusion_model.pt")
torch.save(fold_reports[-1]["model_state"], best_model_path)

# 5) Visualizations
train_curve_path = plot_training_curves(metrics)
cm_path = plot_confusion_matrix(fold_reports[-1]["labels"], fold_reports[-1]["preds"])
roc_path = plot_multiclass_roc(fold_reports[-1]["labels"], fold_reports[-1]["probs"])

# 6) Package artifacts
zip_path = package_outputs(best_model_path, [train_curve_path, cm_path, roc_path])
print("Packaged outputs at:", zip_path)